# 01 数据处理
**任务**：石脑油终馏点软测量 —— 数据探索、清洗、特征工程与特征选择

---
## 📌 背景：什么是软测量（Soft Sensor）？

石脑油**终馏点**（End Point, EP）是蒸馏装置最重要的产品质量指标之一。
传统做法：每隔2~4小时人工采样送化验室，化验结果滞后、离散，无法实时控制。

**软测量**用实时可测的DCS过程变量（温度、压力、流量）**推断**难以连续测量的质量指标：
$$\hat{y}_{终馏点} = f(T_{塔板}, P_{塔顶}, F_{回流}, \ldots)$$

本质是一个**有监督回归问题**，输入~77维过程变量，输出1维终馏点（°C）。

**数据说明**：
- 特征：CDU2 DCS变量（TI温度、TIC温控、FI流量、FIC流控、PI压力、ATI在线分析仪）
- 目标列：`FORMATTED_ENTRY`（化验室终馏点，°C，ASTM D86 End Point）
- **运行顺序**：本notebook → `02_model_training.ipynb` → `03_prediction_evaluation.ipynb`

## 1. 环境初始化

In [ ]:
import sys, warnings, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 100})
sns.set_theme(style='whitegrid')
Path('../outputs/processed').mkdir(parents=True, exist_ok=True)
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
print('环境初始化完成')

In [ ]:
with open('../config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
TARGET_COL  = cfg['data']['target_col']
PRODUCT_COL = cfg['data'].get('product_col')
BAD_THRESH  = cfg['data'].get('bad_value_threshold', -50.0)
SEP         = cfg['data'].get('separator', '\t')
print(f'目标列: {TARGET_COL}  |  坏值阈值: < {BAD_THRESH}  |  分隔符: {repr(SEP)}')

## 2. 加载原始数据

In [ ]:
files = sorted(glob.glob('../data/data_confidence_*.csv'))
if not files:
    raise FileNotFoundError('请将CSV文件放入 data/ 目录')
dfs = []
for f in files:
    df = pd.read_csv(f, sep=SEP, encoding='utf-8-sig')
    dfs.append(df)
    print(f'  {Path(f).name}: {df.shape[0]}行 x {df.shape[1]}列')
data = pd.concat(dfs, ignore_index=True)
print(f'\n合并后: {data.shape[0]} 行 x {data.shape[1]} 列')
data.head(3)

## 3. 探索性数据分析（EDA）

In [ ]:
print('=== 数据类型 ===')
print(data.dtypes.value_counts())
print(f'\n数值列: {data.select_dtypes(include=np.number).shape[1]}  非数值列: {list(data.select_dtypes(exclude=np.number).columns)}')
missing = data.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('\n=== 缺失值（缺失率>0的列）===')
print(missing if len(missing) > 0 else '无缺失值')

In [ ]:
y_raw   = data[TARGET_COL].dropna()
y_valid = y_raw[y_raw > BAD_THRESH]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(y_valid, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(y_valid.mean(), color='red', linestyle='--', label=f'均值={y_valid.mean():.1f}°C')
axes[0].set_title(f'终馏点分布  n={len(y_valid)}')
axes[0].set_xlabel('终馏点 (°C)'); axes[0].legend()
axes[1].plot(y_valid.values, color='steelblue', linewidth=0.8, alpha=0.8)
axes[1].set_title('终馏点时序曲线'); axes[1].set_ylabel('°C')
plt.tight_layout()
plt.savefig('../outputs/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(y_valid.describe().round(2))

In [ ]:
if PRODUCT_COL and PRODUCT_COL in data.columns:
    print('产品类型分布:')
    print(data[PRODUCT_COL].value_counts())
    fig, ax = plt.subplots(figsize=(8, 4))
    for prod, grp in data.groupby(PRODUCT_COL):
        yg = grp[TARGET_COL].dropna(); yg = yg[yg > BAD_THRESH]
        if len(yg) > 0:
            ax.hist(yg, bins=30, alpha=0.6, label=f'PRODUCT={prod} (n={len(yg)})')
    ax.set_title('按产品类型的终馏点分布'); ax.legend()
    plt.show()
else:
    print('无产品类型列')

## 4. DCS坏值识别与处理

> ### 📐 原理：为什么DCS会产生坏值？
> 
> DCS（分布式控制系统）的仪表在以下情况会输出标志值而非真实测量值：
> - 传感器断线、短路、量程超限 → 输出 `-99.9` 或 `-100`（OPC标准坏值标志）
> - 仪表维护中（MANUAL状态）→ 输出固定替代值
> - 网络通讯中断 → 输出最后一次有效值或坏值标志
> 
> **处理策略**：
> - 值 < -50 → 识别为坏值，替换为 `NaN`
> - 后续用中位数插补（中位数对异常值比均值更鲁棒）
> - 目标列出现坏值的行直接删除（不可用样本）

In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
bad_mask  = data[numeric_cols] < BAD_THRESH
bad_count = bad_mask.sum().sort_values(ascending=False)
bad_cols  = bad_count[bad_count > 0]
print(f'含坏值的列数: {len(bad_cols)}  |  坏值总数: {bad_mask.values.sum()}')
print('\nTop-10 坏值最多的列:')
print(bad_cols.head(10))
if len(bad_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    (bad_cols.head(20) / len(data) * 100).plot(kind='bar', ax=ax, color='tomato')
    ax.set_title('坏值占比 Top-20 列 (%)'); ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

In [ ]:
data_clean = data.copy()
data_clean[numeric_cols] = data_clean[numeric_cols].where(
    data_clean[numeric_cols] >= BAD_THRESH, other=np.nan)
valid_mask = data_clean[TARGET_COL].notna()
data_clean = data_clean[valid_mask].reset_index(drop=True)
print(f'有效样本: {len(data_clean)} 行  (删除了 {(~valid_mask).sum()} 行无效目标)')

## 5. 特征统计分析

> ### 📐 原理：为什么要分析特征与目标的相关性？
> 
> **Pearson相关系数** $r = \frac{\text{cov}(X_j, y)}{\sigma_{X_j} \sigma_y}$ 衡量线性关联强度（$r \in [-1, 1]$）
> 
> - $|r| > 0.5$：强相关，该特征对终馏点有直接影响
> - $|r| < 0.1$：弱/无相关，该特征可能噪音居多
> 
> **多重共线性**：蒸馏塔相邻塔板温度（TI_101~TI_135）物理上必然高度相关（$r > 0.95$），
> 保留冗余特征会导致线性模型（PLS、Ridge）系数不稳定，树模型则会分散特征重要度。
> 解决：删除相关性 > 0.97 的冗余列，保留信息量最大的代表特征。

In [ ]:
drop_meta = [TARGET_COL] + ([PRODUCT_COL] if PRODUCT_COL else [])
feat_cols = [c for c in data_clean.select_dtypes(include=np.number).columns if c not in drop_meta]
type_groups = {
    '温度(TI/TIC)':       [c for c in feat_cols if '.TI' in c or '.TIC' in c],
    '流量(FI/FIC/FIQ)':   [c for c in feat_cols if '.FI' in c or '.FIC' in c or '.FIQ' in c],
    '压力(PI/PIC/PDIC)':  [c for c in feat_cols if '.PI' in c or '.PIC' in c or '.PDIC' in c],
    '分析仪(ATI/AI)':     [c for c in feat_cols if '.ATI' in c or '.AI' in c],
}
classified = [c for lst in type_groups.values() for c in lst]
type_groups['其他'] = [c for c in feat_cols if c not in classified]
print('特征类型统计:')
for t, cols in type_groups.items():
    print(f'  {t}: {len(cols)} 个')
print(f'\n特征总数: {len(feat_cols)}')

In [ ]:
X_raw = data_clean[feat_cols]
stats = X_raw.describe().T
stats['missing%'] = (X_raw.isnull().mean() * 100).round(2)
stats['cv'] = (stats['std'] / stats['mean'].abs().replace(0, np.nan)).round(3)
print('变异系数最大的10列（信息量最丰富）:')
print(stats.nlargest(10, 'cv')[['mean', 'std', 'cv', 'missing%']])
print('\n变异系数最小的10列（接近常数，低信息量）:')
print(stats.nsmallest(10, 'cv')[['mean', 'std', 'cv', 'missing%']])

In [ ]:
y_series = data_clean[TARGET_COL]
corr_with_target = X_raw.corrwith(y_series).abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
corr_with_target.head(20).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('与终馏点相关性 Top-20 (|r|)'); axes[0].set_ylabel('|Pearson r|')
axes[0].tick_params(axis='x', rotation=45)
axes[1].hist(corr_with_target, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0.3, color='red', linestyle='--', label='|r|=0.3')
axes[1].set_title('全部特征相关性分布'); axes[1].set_xlabel('|Pearson r|'); axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/feature_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'|r| > 0.5: {(corr_with_target > 0.5).sum()} 个  |  |r| > 0.3: {(corr_with_target > 0.3).sum()} 个')

In [ ]:
top_feat    = corr_with_target.head(30).index.tolist()
corr_matrix = X_raw[top_feat].corr()
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=[c.replace('CDU2.', '') for c in top_feat],
            yticklabels=[c.replace('CDU2.', '') for c in top_feat])
ax.set_title('Top-30特征 相关性矩阵（红=正相关，蓝=负相关）')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 预处理流水线

> ### 📐 原理：各预处理步骤说明
> 
> #### 6.1 低方差过滤
> 方差 $< 0.001$ 的特征几乎是常数，对模型无贡献且增加计算成本，直接删除。
> 
> #### 6.2 异常值处理（IQR方法）
> $$[Q_1 - 3 \times IQR,\; Q_3 + 3 \times IQR]$$ 范围外的值视为异常，截断至边界。
> - 使用3倍IQR而非1.5倍，因工业数据正常波动范围较大
> - **关键**：统计量仅从训练集计算，测试集用训练集统计量处理，防止数据泄露
> 
> #### 6.3 缺失值插补
> 用各列的**中位数**填充剩余NaN（比均值对异常值更鲁棒）
> 
> #### 6.4 高相关冗余删除
> 特征间相关系数 $|r| > 0.97$，删除其中一个（保留与目标更相关的那个）
> 
> #### 6.5 标准化（Z-score）
> $$x' = \frac{x - \mu}{\sigma}$$ 消除量纲差异（温度~300°C vs 流量~100 t/h），
> 对PLS、Ridge、LSTM等模型必须进行，树模型不敏感但统一处理利于比较。

In [ ]:
from src.preprocessor import Preprocessor

X_df   = data_clean[feat_cols].copy()
y      = data_clean[TARGET_COL].copy()
n_total = len(y)
n_train = int(n_total * (1 - cfg['training'].get('test_size', 0.2)))

X_tr_df = X_df.iloc[:n_train]; X_te_df = X_df.iloc[n_train:]
y_train = y.iloc[:n_train].reset_index(drop=True)
y_test  = y.iloc[n_train:].reset_index(drop=True)
print(f'总样本: {n_total}  训练集: {n_train}  测试集: {n_total - n_train}')
print('注意：按时间顺序切分，不做随机打乱（防止未来数据泄露到训练集）')

In [ ]:
cfg['preprocessing']['add_rolling_features'] = False
prep = Preprocessor(cfg)
X_train_prep = prep.fit_transform(X_tr_df.reset_index(drop=True), y_train)

# 测试集：使用训练集统计量变换
X_te_aligned = X_te_df.reset_index(drop=True)
for c in prep.kept_cols_before_rolling:
    if c not in X_te_aligned.columns: X_te_aligned[c] = np.nan
X_te_aligned = X_te_aligned[prep.kept_cols_before_rolling]
if cfg['preprocessing'].get('outlier_method', 'iqr') == 'iqr':
    factor = cfg['preprocessing'].get('outlier_iqr_factor', 3.0)
    lo = prep._q1 - factor * prep._iqr; hi = prep._q3 + factor * prep._iqr
    lo = lo[[c for c in lo.index if c in X_te_aligned.columns]]
    hi = hi[[c for c in hi.index if c in X_te_aligned.columns]]
    X_te_aligned = X_te_aligned.clip(lower=lo, upper=hi, axis=1)
X_te_aligned = X_te_aligned.fillna(prep._fill_values[[c for c in prep._fill_values.index if c in X_te_aligned.columns]])
X_test_prep  = prep.scaler.transform(X_te_aligned)

print(f'预处理后维度 — 训练: {X_train_prep.shape}  测试: {X_test_prep.shape}')
print(f'\n各步骤效果：')
print(f'  原始特征数:          {len(feat_cols)}')
print(f'  低方差过滤删除:      {len(prep.drop_cols_variance)}')
print(f'  高相关冗余删除:      {len(prep.drop_cols_corr)}')
print(f'  最终保留:            {len(prep.kept_cols)}')

## 7. 特征选择

> ### 📐 原理：PLS VIP + XGBoost重要度联合选择
> 
> #### PLS VIP（Variable Importance in Projection）
> $$\text{VIP}_j = \sqrt{p \cdot \frac{\sum_{h=1}^{H} (w_{jh}^2 \cdot SS_h)}{\sum_{h=1}^{H} SS_h}}$$
> 其中 $w_{jh}$ 是第$j$个变量在第$h$个PLS成分中的权重，$SS_h$ 是该成分解释的因变量方差。
> - $\text{VIP} \geq 1.0$：该变量对Y的解释有重要贡献
> - 适合捕捉**线性关系**中的重要变量
> 
> #### XGBoost特征重要度
> 基于**分裂增益**：该特征被用于分裂时带来的目标函数减少量总和。
> - 能捕捉**非线性关系**中的重要变量
> - 与PLS VIP互补
> 
> #### 联合策略
> 两种得分分别归一化到[0,1]后取平均，选Top-K特征，兼顾线性和非线性重要性。

In [ ]:
from src.feature_selector import FeatureSelector

sel    = FeatureSelector(cfg)
X_train = sel.fit_transform(X_train_prep, y_train.values, feature_names=prep.kept_cols)
X_test  = sel.transform(X_test_prep)

selected_names = [prep.kept_cols[i] for i in sel.selected_indices if i < len(prep.kept_cols)]
print(f'特征选择后 — 训练: {X_train.shape}  测试: {X_test.shape}')
print('\n选中的特征（按得分排序）:')
for i, name in enumerate(selected_names[:15], 1):
    score = sel.feature_scores[sel.selected_indices[i-1]]
    print(f'  {i:2d}. {name.replace("CDU2.",""):20s}  score={score:.4f}')

In [ ]:
scores_sorted = sel.feature_scores[sel.selected_indices]
fig, ax = plt.subplots(figsize=(8, max(6, len(selected_names) * 0.35)))
ax.barh(range(len(selected_names)), scores_sorted[::-1], color='steelblue')
ax.set_yticks(range(len(selected_names)))
ax.set_yticklabels([n.replace('CDU2.', '') for n in selected_names[::-1]], fontsize=8)
ax.set_title(f'联合特征重要度得分 (PLS VIP + XGBoost, Top-{len(selected_names)})')
ax.set_xlabel('归一化得分')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_selection_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 保存处理结果

In [ ]:
import joblib
np.save('../outputs/processed/X_train.npy', X_train)
np.save('../outputs/processed/X_test.npy',  X_test)
np.save('../outputs/processed/y_train.npy', y_train.values)
np.save('../outputs/processed/y_test.npy',  y_test.values)
joblib.dump(selected_names, '../outputs/processed/selected_names.pkl')
joblib.dump(prep,           '../outputs/processed/preprocessor.pkl')
joblib.dump(sel,            '../outputs/processed/feature_selector.pkl')
print('保存完成:')
for f in sorted(Path('../outputs/processed').iterdir()):
    print(f'  {f.name}')
print('\n➡  请运行 02_model_training.ipynb')